# 05 · Pairwise Evaluation with a Kaggle Judge Model
**PA2 Task 7 (Sec. 13).** No GPU needed. This notebook must be a **Kaggle Benchmarks** notebook so that `kaggle_benchmarks` can call judge models such as `google/gemini-3.6-flash` for free (Sec. 13.1).

Attach as inputs: the saved outputs of notebook **04** (pairs for DPO vs SFT) and notebook **03** (persona pairs).

Reliability (Sec. 13.2): 5 pairs per request, ≥5 s sleep before every attempt, up to 3 retries, failures logged to `*_failures.jsonl`, judgments appended after each batch, and `--reuse_judgments` resumes without re-spending calls.

## Step 0 · Copy the code into Kaggle's working folder  (PA2 Task 1, Sec. 7 + Sec. 5.4)
Notebook settings (right panel): **Accelerator = None**, **Internet = On** (needs a phone-verified Kaggle account).

The code comes from your uploaded Kaggle Dataset (the `small-model-alignment.zip`). Attach it with **+ Add Input → Datasets → Your Datasets**. This cell finds it automatically, so there are no paths to edit.

In [ ]:
# =============================================================================
# Step 0: copy the code from the attached Kaggle Dataset into /kaggle/working
#   /kaggle/working is the ONLY folder Kaggle keeps when you "Save Version" (PA2 Sec. 5.5)
# =============================================================================
import os, shutil, glob

REPO_DIR = "/kaggle/working/small-model-alignment"

def find_code_dataset():
    # Locate the uploaded code dataset anywhere under /kaggle/input.
    # Attached notebook OUTPUTS also contain a copy of the repo, but those copies
    # have an outputs/ folder; the clean code dataset does not, so prefer it.
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)]
    clean = [h for h in hits if not os.path.isdir(os.path.join(h, "outputs"))]
    if not (clean or hits):
        raise FileNotFoundError("Code dataset not found: + Add Input -> Datasets -> Your Datasets -> your code dataset")
    return (clean or hits)[0]

if not os.path.exists(os.path.join(REPO_DIR, "submission_adapter.py")):
    src = find_code_dataset()
    print("copying code from", src)
    shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
    !chmod -R u+w {REPO_DIR}   # /kaggle/input is read-only; make the copy writable
%cd {REPO_DIR}

# Memory/perf environment for the T4s, inherited by every `!python` / `!torchrun` below:
#   expandable_segments -> less CUDA fragmentation with dynamic (per-batch) padding
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

def sh(cmd):
    # Run a shell command, stream its output here, and STOP the notebook if it fails
    # (so a crashed torchrun does not silently continue during Save & Run All).
    get_ipython().system(cmd)
    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"command failed: {cmd[:150]}")
!ls

## Step 1 · Copy the saved pairwise JSONL files in (judging reads saved files only; nothing is regenerated, Sec. 13.1 item 4)

In [ ]:
# =============================================================================
# Step 1: collect pairs files from the attached inputs
# =============================================================================
os.makedirs("outputs/eval", exist_ok=True)
for f in glob.glob("/kaggle/input/**/outputs/eval/*_pairs.jsonl", recursive=True):
    shutil.copy(f, "outputs/eval/")
!wc -l outputs/eval/*_pairs.jsonl

## Step 2 · Check the judge API
`kaggle_benchmarks` is preinstalled in Benchmarks notebooks. This cell lists what it exposes so you can confirm the model name. If `google/gemini-3.6-flash` is not listed, pick another supported judge and record it in the report (Sec. 13).

In [ ]:
# =============================================================================
# Step 2: inspect kaggle_benchmarks and send one tiny test prompt
# =============================================================================
import kaggle_benchmarks as kbench
print([a for a in dir(kbench) if not a.startswith("_")])
try:
    print("models:", list(kbench.llms.keys()) if hasattr(kbench.llms, "keys") else kbench.llms)
except Exception as e:
    print("could not list models:", e)
JUDGE = "google/gemini-3.6-flash"
print(kbench.llms[JUDGE].prompt("Reply with exactly: OK"))

## Step 3 · Dry run (no API calls) to test the plumbing

In [ ]:
# =============================================================================
# Step 3: dry run with a fake judge that always says "tie"
# =============================================================================
sh("python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_uf_vs_sft_pairs.jsonl "
   "--output_path outputs/eval/_dry_judged.jsonl --dry_run --sleep_seconds 0 | tail -n 5")
!rm -f outputs/eval/_dry_*

## Step 4 · Judge the three comparisons
The prompt follows RA2 App. C.2 (one-sentence comparison, then a single verdict) and tells the judge not to reward length, the bias RA2 Sec. 6.4 reports for GPT-4. HH uses the safety rubric (reward appropriate refusal, privacy, calibrated uncertainty, safe redirection — Sec. 13.2). The persona comparison gets two verdicts: **style** and **quality**.

In [ ]:
# =============================================================================
# Step 4: real judging (resumable - just re-run the cell after a disconnect)
# =============================================================================
JOBS = [("dpo_uf_vs_sft", "general", "dpo_uf"),
        ("dpo_hh_vs_sft", "safety", "dpo_hh"),
        ("lora_vs_base_persona", "persona", "lora_dpo")]
for name, mode, focus in JOBS:
    if not os.path.exists(f"outputs/eval/{name}_pairs.jsonl"):
        print("missing pairs:", name); continue
    sh(f"python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/{name}_pairs.jsonl "
       f"--output_path outputs/eval/{name}_judged.jsonl --judge_model {JUDGE} --mode {mode} "
       f"--focus_model {focus} --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments")

## Step 5 · Results table for the report  (PA2 Sec. 14.5)

In [ ]:
# =============================================================================
# Step 5: summary table (wins / ties / losses / unknown / failures, win rate with and without ties)
# =============================================================================
import json, pandas as pd
rows = []
for f in sorted(glob.glob("outputs/eval/*_summary.json")):
    s = json.load(open(f))
    for key in ("winner", "style", "quality"):
        if key in s:
            d = s[key]
            rows.append({"comparison": os.path.basename(f).replace("_judged_summary.json", ""), "verdict": key,
                         "n": s["n_pairs"], "wins": d["wins"], "ties": d["ties"], "losses": d["losses"],
                         "unknown": d["unknown"], "failures": d["failures"],
                         "win_rate_excl_ties": d["win_rate_excluding_ties"], "win_rate_ties_half": d["win_rate_ties_as_half"],
                         "judge": s["judge_model"]})
df = pd.DataFrame(rows); df.to_csv("outputs/eval/results_table.csv", index=False); df

In [ ]:
# Per-category breakdown for the HH set (where did DPO help or hurt? Sec. 14.6)
s = json.load(open("outputs/eval/dpo_hh_vs_sft_judged_summary.json"))
pd.DataFrame(s["winner"]["per_category"]).T.fillna(0).astype(int)

In [ ]:
!zip -q -r /kaggle/working/judge_outputs.zip outputs/eval

## Final step · Preserve the outputs  (PA2 Sec. 5.5)
1. Click **Save Version → Save & Run All (Commit)** (or, if you ran interactively, *Quick Save* keeps the current outputs).
2. After the version finishes, its `/kaggle/working` folder becomes an **Output** you can attach to the next notebook with **Add Input → Your Work → Notebooks**.